# Treino dos quatro modelos de LIBRAS no Google Colab

Este notebook refaz, do zero, o treino dos quatro classificadores do projeto
[LibrasMediaPipe](https://github.com/RuBeNsMoTa13/LibrasMediaPipe) e anota tudo o que foi usado:

1. **MediaPipe `.task`**, treinado com o MediaPipe Model Maker (aprendizado por transferência).
2. **Random Forest, SVM e SNN**, treinados com `src/treino/treinar_modelos.py` sobre o resumo de 128 números do `.task`.

As fotos são o dataset inteiro do Kaggle
[williansoliveira/libras](https://www.kaggle.com/datasets/williansoliveira/libras) (46.262 fotos),
baixado com o `kagglehub`. As fotos não ficam no GitHub.
Os quatro modelos usam a mesma divisão do dataset: a pasta `train` para treinar e a pasta `test` para testar.
Nenhuma foto de teste entra no treino.

**Como usar:** menu *Ambiente de execução > Executar tudo*. No fim, o Colab baixa o arquivo
`treino_libras.zip` com os modelos novos, os registros do treino e os gráficos.

## 1. Instalar o MediaPipe Model Maker

In [ ]:
!pip install --upgrade pip
!pip install mediapipe-model-maker

## 2. Baixar o projeto do GitHub

Os scripts de treino e avaliação vêm do projeto.

In [ ]:
BRANCH = "main"

!git lfs install
!rm -rf /content/LibrasMediaPipe
!git clone --depth 1 -b {BRANCH} https://github.com/RuBeNsMoTa13/LibrasMediaPipe.git /content/LibrasMediaPipe
%cd /content/LibrasMediaPipe
!git log -1 --format="Versão do projeto: %h (%ad)" --date=short

## 3. Baixar o dataset do Kaggle

Baixa as 46.262 fotos com o `kagglehub` e conta quantas há em cada pasta.

In [ ]:
import os
from pathlib import Path

import kagglehub

pasta_kaggle = Path(kagglehub.dataset_download("williansoliveira/libras"))
# Acha a pasta que contém train/ e test/ dentro do download
raiz_dados = next(p.parent for p in sorted(pasta_kaggle.rglob("train")) if (p.parent / "test").is_dir())
FOTOS_KAGGLE = {parte: sum(1 for p in (raiz_dados / parte).rglob("*") if p.is_file())
                for parte in ("train", "test")}
print("Dataset em:", raiz_dados)
print("Fotos:", FOTOS_KAGGLE)

# Os scripts do projeto leem as fotos desta pasta
os.environ["LIBRAS_DADOS"] = str(raiz_dados)

## 4. Configuração do treino do `.task`

Escolha o `PRESET`:

- `"longo"`: 200 épocas com lote de 32 fotos. A cabeça do `.task` aprende até estabilizar.
- `"padrao"`: o padrão do Model Maker (10 épocas, lote de 2), igual ao treino original.

Os outros valores são os padrões do Model Maker e ficam anotados no registro.

In [ ]:
PRESET = "longo"   # "longo" ou "padrao"
SEED = 42
PROPORCAO_VALIDACAO = 0.1   # 10% das fotos de treino separadas para validação

PRESETS = {
    "longo":  dict(learning_rate=0.001, batch_size=32, epochs=200),
    "padrao": dict(learning_rate=0.001, batch_size=2, epochs=10),
}
HPARAMS = dict(**PRESETS[PRESET], lr_decay=0.99, gamma=2, shuffle=True)
MODEL_OPTIONS = dict(dropout_rate=0.05, layer_widths=[])
print("Preset:", PRESET)
print("HParams:", HPARAMS)
print("ModelOptions:", MODEL_OPTIONS)

## 5. Preparar as pastas do Model Maker

O Model Maker exige uma classe `none` ("nenhuma letra"). Como o dataset não tem fotos sem letra,
ela fica como uma pasta vazia, igual ao treino original.

In [ ]:
import shutil

DADOS_TASK = Path("/content/dados_task")
shutil.rmtree(DADOS_TASK, ignore_errors=True)
for origem, destino in [("train", "treino"), ("test", "teste")]:
    shutil.copytree(raiz_dados / origem, DADOS_TASK / destino)
    (DADOS_TASK / destino / "none").mkdir()
    letras = sorted(p.name for p in (DADOS_TASK / destino).iterdir() if p.is_dir())
    fotos = sum(1 for p in (DADOS_TASK / destino).rglob("*.png"))
    print(f"{destino}: {fotos} fotos em {len(letras)} pastas -> {letras}")

## 6. Achar a mão em cada foto

O Model Maker passa cada uma das 46 mil fotos pelo detector de mão do Google. Fotos em que a mão não é encontrada ficam de fora.
É uma das etapas mais demoradas.

In [ ]:
import random

import tensorflow as tf
from mediapipe_model_maker import gesture_recognizer

tf.keras.utils.set_random_seed(SEED)   # mesma semente para random, numpy e TensorFlow
random.seed(SEED)

preproc = gesture_recognizer.HandDataPreprocessingParams()
dados_treino = gesture_recognizer.Dataset.from_folder(dirname=str(DADOS_TASK / "treino"), hparams=preproc)
dados_teste = gesture_recognizer.Dataset.from_folder(dirname=str(DADOS_TASK / "teste"), hparams=preproc)
treino, validacao = dados_treino.split(1 - PROPORCAO_VALIDACAO)

print("Classes:", dados_treino.label_names)
print(f"Mãos encontradas: treino {len(treino)} | validação {len(validacao)} | teste {len(dados_teste)}")

## 7. Treinar o `.task`

In [ ]:
hparams = gesture_recognizer.HParams(export_dir="exported_model", **HPARAMS)
model_options = gesture_recognizer.ModelOptions(**MODEL_OPTIONS)
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams, model_options=model_options)
modelo = gesture_recognizer.GestureRecognizer.create(
    train_data=treino, validation_data=validacao, options=options)

## 8. Testar, exportar e anotar o treino do `.task`

O `.task` novo substitui `models/gesture_recognizer.task`, e o registro do treino vai para
`results/treinos/task.json` (é dele que saem as Figuras 1 e 2 do TCC).

In [ ]:
import json
import platform
from datetime import datetime

perda_teste, acc_teste = modelo.evaluate(dados_teste, batch_size=1)
print(f"Teste (fotos de data/libras/test com mão): perda {perda_teste:.4f} | acurácia {acc_teste:.4f}")

modelo.export_model()
shutil.copy("exported_model/gesture_recognizer.task", "models/gesture_recognizer.task")
print("Modelo salvo em models/gesture_recognizer.task")

h = modelo._history.history
chave_acc = next(k for k in h if "acc" in k and not k.startswith("val_"))
historico = [{"epoca": i + 1,
              "perda_treino": round(float(h["loss"][i]), 6),
              "acuracia_treino": round(float(h[chave_acc][i]), 6),
              "perda_validacao": round(float(h["val_loss"][i]), 6),
              "acuracia_validacao": round(float(h["val_" + chave_acc][i]), 6)}
             for i in range(len(h["loss"]))]

def versao(pacote):
    try:
        from importlib.metadata import version
        return version(pacote)
    except Exception:
        return None

versao_projeto = !git rev-parse --short HEAD

registro = {
    "descricao": "Treino do models/gesture_recognizer.task feito com notebooks/treinar_no_colab.ipynb no Google Colab.",
    "data": datetime.now().astimezone().isoformat(timespec="seconds"),
    "versao_do_projeto": versao_projeto[0],
    "dataset": "Kaggle williansoliveira/libras, dataset inteiro baixado com kagglehub",
    "fotos": FOTOS_KAGGLE,
    "divisao": (f"pasta train do Kaggle: {1 - PROPORCAO_VALIDACAO:.0%} treino e "
                f"{PROPORCAO_VALIDACAO:.0%} validação (sorteio com semente {SEED}); pasta test só para o teste"),
    "semente": SEED,
    "preset": PRESET,
    "hparams": HPARAMS,
    "model_options": MODEL_OPTIONS,
    "classes": list(dados_treino.label_names),
    "maos_encontradas": {"treino": len(treino), "validacao": len(validacao), "teste": len(dados_teste)},
    "historico": historico,
    "teste": {"perda": round(float(perda_teste), 6), "acuracia": round(float(acc_teste), 6)},
    "versoes": {"python": platform.python_version(),
                **{p: versao(p) for p in ["mediapipe-model-maker", "mediapipe", "tensorflow"]}},
}
Path("results/treinos").mkdir(parents=True, exist_ok=True)
Path("results/treinos/task.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print("Registro salvo em results/treinos/task.json")

## 9. Treinar Random Forest, SVM e SNN

Os três usam o resumo de 128 números que o `.task` calcula para cada mão, com a mesma divisão train/test do Kaggle
(a variável `LIBRAS_DADOS`, definida na etapa 3, aponta os scripts para a pasta baixada).
O cache antigo é apagado porque guarda as respostas do `.task` anterior. Com 46 mil fotos, o SVM é a parte mais lenta.
O registro deste treino vai para `results/treinos/classificadores.json`.

In [ ]:
!pip install -q snntorch seaborn
!rm -f results/tables/resumo_libras_kaggle.npz
!python src/treino/treinar_modelos.py

## 10. Gerar os gráficos do TCC

In [ ]:
%env MPLBACKEND=Agg
!python src/avaliacao/curvas_treino_task.py
!python src/avaliacao/matriz_confusao_task.py
!python src/avaliacao/graficos_modelos.py
!python src/avaliacao/grafico_tsne.py

## 11. Baixar o resultado

O arquivo `treino_libras.zip` tem os quatro modelos (`models/`), os registros do treino
(`results/treinos/`) e os gráficos (`results/figures/`). Mande esse zip na conversa do projeto,
ou copie as pastas para dentro do projeto no seu computador.

In [ ]:
from google.colab import files

!rm -f treino_libras.zip
!zip -r treino_libras.zip models results/treinos results/figures
files.download("treino_libras.zip")